# Google Health API — steps extraction (migration test)

**This is the correct migration target** for the Wandern Eric backend, replacing the
Fitbit Web API.

- The old `gh_migration.ipynb` / `google_health_example.py` use the **Google Fit REST API**
  (`fitness/v1`), which is **shut down at the end of 2026** and is *not* the same thing as
  "Health Connect" (which is Android, on-device only — no cloud API).
- The **Google Health API** (`https://health.googleapis.com/v4`) is the server-to-server
  REST successor to the Fitbit Web API. Since Eric is coming *from* Fitbit, this is one clean
  destination.

### Prerequisites (do these first)
1. In Google Cloud Console: enable the **Google Health API** for your project. (It may require
   requesting access / allowlisting — check https://developers.google.com/health/setup .)
2. Create OAuth 2.0 credentials. Google's setup recommends a **Web Server** client with redirect
   `https://www.google.com`; for this local notebook test we reuse the installed-app flow with a
   `localhost` redirect, so add `http://localhost:8080/` as an authorized redirect URI too.
3. Connect Eric's Fitbit (or Pixel) account to Google Health so data actually flows in.
4. `poetry install` already covers `google-auth`, `google-auth-oauthlib`, `requests`.

Docs: setup https://developers.google.com/health/setup · endpoints
https://developers.google.com/health/endpoints · dailyRollUp
https://developers.google.com/health/reference/rest/v4/users.dataTypes.dataPoints/dailyRollUp

In [ ]:
import json
import datetime
from pathlib import Path

import google.oauth2.credentials
import google_auth_oauthlib.flow
from google.auth.transport.requests import AuthorizedSession, Request

In [ ]:
ROOT_DIR = Path().resolve()
CLIENT_SECRET_PATH = ROOT_DIR / "secrets/client_secret_health.json"  # the Google Health OAuth client
TOKEN_PATH = ROOT_DIR / "secrets/token_health.json"

BASE_URL = "https://health.googleapis.com/v4"

# Read-only access to activity & fitness data (includes steps).
SCOPES = ["https://www.googleapis.com/auth/googlehealth.activity_and_fitness.readonly"]

## Auth

Same OAuth pattern as the Fit notebook, just pointed at the Google Health scope. Produces
credentials that `AuthorizedSession` will auto-refresh on each call.

In [ ]:
def authenticate():
    """Authenticate via OAuth2, reusing the saved token when possible."""
    creds = None

    if TOKEN_PATH.exists():
        with open(TOKEN_PATH) as f:
            token_data = json.load(f)
        creds = google.oauth2.credentials.Credentials(
            token=token_data.get("token"),
            refresh_token=token_data.get("refresh_token"),
            token_uri=token_data.get("token_uri"),
            client_id=token_data.get("client_id"),
            client_secret=token_data.get("client_secret"),
            scopes=SCOPES,
        )

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = google_auth_oauthlib.flow.InstalledAppFlow.from_client_secrets_file(
                str(CLIENT_SECRET_PATH), scopes=SCOPES
            )
            creds = flow.run_local_server(port=8080)

        token_data = {
            "token": creds.token,
            "refresh_token": creds.refresh_token,
            "token_uri": creds.token_uri,
            "client_id": creds.client_id,
            "client_secret": creds.client_secret,
        }
        TOKEN_PATH.parent.mkdir(parents=True, exist_ok=True)
        with open(TOKEN_PATH, "w") as f:
            json.dump(token_data, f)

    return creds

In [ ]:
creds = authenticate()
creds

## The dailyRollUp request

`POST /users/me/dataTypes/steps/dataPoints:dailyRollUp` aggregates steps into civil-day
buckets. `windowSizeDays=1` => one bucket per day. The time range uses **CivilDateTime**
objects (local time, no timezone): `{"date": {year, month, day}, "time": {hours, minutes}}`.
The range is half-open, so one day = `[date 00:00, date+1 00:00)`.

In [ ]:
def _civil(d: datetime.date, hours: int = 0, minutes: int = 0) -> dict:
    """Build a Google Health CivilDateTime for the start of a day (local time)."""
    return {
        "date": {"year": d.year, "month": d.month, "day": d.day},
        "time": {"hours": hours, "minutes": minutes},
    }


def daily_rollup_steps(session: AuthorizedSession, start_date: datetime.date,
                       end_date: datetime.date, window_size_days: int = 1) -> dict:
    """Raw dailyRollUp response for steps over [start_date, end_date)."""
    url = f"{BASE_URL}/users/me/dataTypes/steps/dataPoints:dailyRollUp"
    body = {
        "range": {"start": _civil(start_date), "end": _civil(end_date)},
        "windowSizeDays": window_size_days,
    }
    resp = session.post(url, json=body, headers={"Accept": "application/json"})
    resp.raise_for_status()
    return resp.json()

In [ ]:
# --- Probe one day and print the RAW response ---
# Run this first and eyeball the shape. The parsing below assumes:
#   rollupDataPoints[].civilStartTime.date.{year,month,day}
#   rollupDataPoints[].steps.countSum  (a string, e.g. "3822")
# If the real response differs, adjust parse_daily_steps() to match what you see here.
session = AuthorizedSession(creds)

probe_day = datetime.date(2026, 2, 26)
raw = daily_rollup_steps(session, probe_day, probe_day + datetime.timedelta(days=1))
print(json.dumps(raw, indent=2))

## Parse → the existing `{date, count}` record shape

Keeping the same record shape Fitbit produced means `data.json`, the aggregations, and the
frontend all keep working unchanged.

In [ ]:
def parse_daily_steps(rollup_json: dict) -> list[dict]:
    """rollupDataPoints -> [{date, count}], one per civil day."""
    records = []
    for rp in rollup_json.get("rollupDataPoints", []):
        d = rp["civilStartTime"]["date"]
        date_str = f"{d['year']:04d}-{d['month']:02d}-{d['day']:02d}"
        count = int(rp.get("steps", {}).get("countSum", 0))
        records.append({"date": date_str, "count": count})
    return records


parse_daily_steps(raw)

## Drop-in replacement for `FitbitController.get_daily_steps`

Same signature/return contract as the Fitbit version in `main.py`, so swapping the controller
is the only change the production pipeline needs.

In [ ]:
def get_daily_steps(session: AuthorizedSession, date: datetime.date) -> dict:
    """Single-day steps total from the Google Health API."""
    raw = daily_rollup_steps(session, date, date + datetime.timedelta(days=1), window_size_days=1)
    records = parse_daily_steps(raw)
    if records:
        return records[0]
    return {"date": date.strftime("%Y-%m-%d"), "count": 0}

In [ ]:
yesterday = datetime.date.today() - datetime.timedelta(days=1)
print(f"Fetching steps for: {yesterday}")
get_daily_steps(session, yesterday)

In [ ]:
# Backfill / historical: one call can roll up many days at once (windowSizeDays=1, wide range).
start = datetime.date.today() - datetime.timedelta(days=7)
end = datetime.date.today()
week = daily_rollup_steps(session, start, end, window_size_days=1)
parse_daily_steps(week)

## Notes for production (`main.py`)

- **Swap the controller, keep the contract.** Replace `FitbitController` with a
  `GoogleHealthController` whose `get_daily_steps(date)` returns `{date, count}`. Everything
  downstream (`update_and_save_fitbit_data`, `calculate_aggregations`, GCS upload, the React
  frontend) stays the same.
- **Intraday (`--intraday`).** Call `daily_rollup_steps(session, today, today+1)` for today's
  cumulative total — same as the Fitbit intraday path. (For finer granularity there's also
  `GET /users/me/dataTypes/steps/dataPoints?filter=...`.)
- **`sedentary_minutes` gap.** The Fitbit record also carried `sedentary_minutes`. That isn't part
  of the `steps` data type here; decide whether to drop it or pull a separate Google Health data
  type. The frontend currently only uses `count`, so dropping it is safe.
- **Tokens.** Google refresh tokens don't single-use-rotate like Fitbit's, so the
  two-machine desync problem goes away — but still run extraction from one place (the Pi).
- **Auth on the Pi.** `run_local_server` needs a browser. Do the first OAuth on a machine with a
  browser, then copy `secrets/token_health.json` to the Pi (it refreshes headlessly after that).
- **Don't commit** `secrets/` or `token_health.json` — already covered by `.gitignore`.

Docs: about https://developers.google.com/health/about · data types
https://developers.google.com/health/data-types · scopes
https://developers.google.com/health/scopes